# CND-MNE walkthrough

This is the tiny bundled example, not a real subject. Two trials, four EEG channels (`Fz`, `Cz`, `Pz`, `Oz`), plus a speech envelope and word onsets.

Run it from the repo root (or from `examples/` — the next cell finds the data either way).

```bash
uv sync --extra dev
uv run jupyter notebook examples/walkthrough.ipynb
```

If `jupyter` is missing: `uv pip install jupyter`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from cnd_mne import inspect_cnd, read_cnd, read_cnd_mne

here = Path.cwd()
root = here if (here / "tests/data/minimal-cnd").exists() else here.parent
source = root / "tests/data/minimal-cnd"
scratch = root / "examples/_scratch"
scratch.mkdir(exist_ok=True)

print("CND folder:", source)
print(sorted(p.name for p in source.iterdir()))

## What those two files are

- `dataSub1.mat` — EEG for subject 1, already cut into trials
- `dataStim.mat` — what was happening on the same trials (envelope, word onsets)

`inspect` only reads MATLAB. It does not need a unit.

In [ ]:
import json

cnd = read_cnd(source, subject=1)
summary = inspect_cnd(cnd)
print(json.dumps({
    "n_trials": summary["n_trials"],
    "channels": summary["neural"]["n_channels"],
    "trial_shapes": summary["neural"]["trial_shapes"],
    "unit": summary["neural"]["data_unit"],
    "features": summary["stimulus"]["feature_names"],
}, indent=2))

## How the converter sits in the middle

MNE wants one continuous recording (`Raw`). CND trials can be different lengths, and the speech envelope is not an EEG channel. So we do **not** dump the whole experiment into one `Raw`.

1. Read the `.mat` files (v5 or v7.3).
2. Keep a Python copy of the CND (`CNDRecording`): trials, stimulus tracks, leftover MATLAB fields.
3. Hand each trial to MNE as its own `Raw`.
4. Hang the leftover CND on `rec.cnd` so write-back still has the envelope.

This example already declares `uV`. Real public files often do not — then you must pass `neural_unit="uV"` (or whatever the owner confirmed).

In [ ]:
rec = read_cnd_mne(source, subject=1)

print("trials:", len(rec.raws))
for i, raw in enumerate(rec.raws, start=1):
    print(f"  trial {i}: {raw.n_times} samples, {raw.info['sfreq']} Hz, {raw.ch_names}")

print("stimulus tracks on rec.cnd:", rec.cnd.stimulus.names)
print("first trial EEG shape (channels x time):", rec.raws[0].get_data().shape)

`rec.raws[0]` is ordinary MNE. `rec.cnd` is everything else. You plot and filter the `Raw`. You need `rec.cnd` when you want MATLAB back.

## Look at the EEG

Trial 1, all four channels. This is fake data, so it will not look like a real scalp recording.

In [ ]:
raw = rec.raws[0]
data = raw.get_data()  # volts, which is what MNE stores
times = raw.times

fig, ax = plt.subplots(figsize=(8, 3))
for i, name in enumerate(raw.ch_names):
    ax.plot(times, data[i] * 1e6, label=name)  # plot in µV
ax.set_xlabel("time (s)")
ax.set_ylabel("µV")
ax.set_title("trial 1 EEG")
ax.legend(loc="upper right")
fig.tight_layout()
plt.show()

In [ ]:
spectrum = rec.raws[0].compute_psd(fmin=1, fmax=40, verbose="error")
fig = spectrum.plot(show=False, amplitude=False)
fig.suptitle("trial 1 power spectrum")
fig.tight_layout()
plt.show()

## Look at the stimulus tracks

These are **not** EEG channels. They live on `rec.cnd` and we ask for an MNE view when we want to plot them.

In [ ]:
envelope = rec.stimulus_raws("Speech Envelope")[0]
fig, ax = plt.subplots(figsize=(8, 2.5))
ax.plot(envelope.times, envelope.get_data()[0])
ax.set_xlabel("time (s)")
ax.set_title("trial 1 speech envelope")
fig.tight_layout()
plt.show()

In [ ]:
words = rec.stimulus_raws("Word Onsets")[0]
fig, ax = plt.subplots(figsize=(8, 2.5))
ax.plot(words.times, words.get_data()[0], drawstyle="steps-post")
ax.set_xlabel("time (s)")
ax.set_title("trial 1 word onsets (impulse track)")
fig.tight_layout()
plt.show()

ann = rec.stimulus_annotations("Word Onsets")[0]
print("same onsets as MNE annotations:", list(zip(ann.onset, ann.description)))

## Do something in MNE, then write CND back

Filter each trial in place. Trial lengths stay the same, which is what write-back needs.

We write through `rec`, not a bare `Raw`, so the envelope comes with us.

In [ ]:
before = rec.raws[0].get_data().copy()

for trial_raw in rec.raws:
    trial_raw.filter(1.0, 15.0, verbose="error")

after = rec.raws[0].get_data()
print("trial 1 changed after filter:", not np.allclose(before, after))

paths = rec.write_cnd(scratch / "filtered-cnd", subject=1, output_unit="uV", overwrite=True)
print("wrote:", paths.neural.name, "and", paths.stimulus.name)

In [ ]:
back = read_cnd_mne(scratch / "filtered-cnd", subject=1)
print("trials still:", len(back.raws))
print("envelope still there:", back.cnd.stimulus.names)
print(
    "filtered EEG survived MATLAB:",
    np.allclose(back.raws[0].get_data(), rec.raws[0].get_data(), atol=1e-12),
)

## Optional: glue trials for a continuous plot

This is **opt-in**. The joins are fake. MNE marks them. Do not treat the glued recording as one real take.

In [ ]:
continuous = rec.concatenate()
print("glued length (samples):", continuous.n_times)
print("boundary annotations:", list(continuous.annotations.description))

## On real lab data

Same calls, different folder:

```python
rec = read_cnd_mne("/path/to/dataCND", subject=1, neural_unit="uV")
```

Only pass a unit the data owner actually confirmed. This package will not guess µV vs volts.